# Mortgage (TBA) Options in LUSID

| Section | Topic |
|---|---|
| 1 | Instrument creation |
| 2 | Recipe |
| 3 | Transaction types |
| 4 | Portfolio and transactions |
| 5 | Valuation |
| 6 | Instrument events |

## The instrument

A mortgage option is an OTC option on a **TBA** (To-Be-Announced) forward contract for generic
agency mortgage-backed securities. Building one means building two instruments, one pointing at
the other:

    underlying   the TBA itself, as a ToBeAnnounced -- upserted first, on its own
    option       a ToBeAnnouncedOption whose `underlying` points back at that TBA

A `ToBeAnnounced` has no coupon cashflows, accrual, or factor to track -- it's just valued as
quantity x price off an EOD quote. The option links to it through a `MasteredInstrument`, so the
underlying has to exist in the instrument master before you can build the option.

`ToBeAnnouncedOption` also asks for a `premium` field. Under `SimpleStatic`, used in this example, the position is valued from its
own quoted mark and `premium` plays no part in that number.

---
## Setup

In [ ]:
import os
import json
import certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())

from datetime import datetime, timezone, timedelta
import pandas as pd

import lusid
import lusid.models as m
from lusid.extensions import (
    SyncApiClientFactory, SecretsFileConfigurationLoader, EnvironmentVariablesConfigurationLoader)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.options.display.float_format = "{:,.2f}".format

SECRETS_PATH = os.getenv("FBN_SECRETS_PATH") or (
    "secrets.json" if os.path.exists("secrets.json") else None)
config_loaders = ([SecretsFileConfigurationLoader(SECRETS_PATH)] if SECRETS_PATH
                   else [EnvironmentVariablesConfigurationLoader()])

factory = SyncApiClientFactory(config_loaders=config_loaders)


def api(cls):
    return factory.build(cls)


instruments_api   = api(lusid.InstrumentsApi)
txn_portfolio_api = api(lusid.TransactionPortfoliosApi)
portfolios_api    = api(lusid.PortfoliosApi)
quotes_api        = api(lusid.QuotesApi)
recipes_api       = api(lusid.ConfigurationRecipeApi)
aggregation_api   = api(lusid.AggregationApi)

meta = api(lusid.ApplicationMetadataApi).get_lusid_versions()
href = meta.links[0].href
print("Domain      :", href[:href.find("/app/")] if "/app/" in href else href)
print("API version :", meta.build_version)

Domain      : https://fbn-tejan.lusid.com
API version : 0.6.16597.0


---
## Configuration

The position gets reported at its own quoted mark, using `SimpleStatic`.

In [2]:
def d(year, month, day):
    return datetime(year, month, day, tzinfo=timezone.utc)


def upsert(key, name, client_internal, definition):
    """Upsert one instrument and return its LUID."""
    resp = instruments_api.upsert_instruments(scope=SCOPE, request_body={
        key: m.InstrumentDefinition(
            name=name,
            identifiers={"ClientInternal": m.InstrumentIdValue(value=client_internal)},
            definition=definition)})
    assert not resp.failed, list(resp.failed.values())[0].detail
    return resp.values[key].lusid_instrument_id


def mastered(luid):
    """Reference an instrument that already exists in the master."""
    return m.MasteredInstrument(
        instrument_type="MasteredInstrument",
        identifiers={"Instrument/default/LusidInstrumentId": luid})


def recreate_portfolio(code, display_name, base_currency, created, recipe=None):
    """Create the portfolio, replacing any earlier run so the book starts empty."""
    request = m.CreateTransactionPortfolioRequest(
        display_name=display_name, code=code, base_currency=base_currency,
        created=created, instrument_scopes=[SCOPE],
        instrument_event_configuration=None if recipe is None else
        m.InstrumentEventConfiguration(
            transaction_template_scopes=["default"],
            recipe_id=m.ResourceId(scope=SCOPE, code=recipe)))
    try:
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Created {SCOPE}/{code}")
    except lusid.ApiException as e:
        if "PortfolioWithIdAlreadyExists" not in str(getattr(e, "body", "")):
            raise
        portfolios_api.delete_portfolio(scope=SCOPE, code=code)
        txn_portfolio_api.create_portfolio(
            scope=SCOPE, create_transaction_portfolio_request=request)
        print(f"Recreated {SCOPE}/{code}")


def upsert_price(luid, price, effective, currency):
    """One Price/mid quote, keyed on the instrument's LUID."""
    quotes_api.upsert_quotes(scope=SCOPE, request_body={
        f"{luid}-{effective:%Y%m%d}": m.UpsertQuoteRequest(
            quote_id=m.QuoteId(
                quote_series_id=m.QuoteSeriesId(
                    provider="Lusid", instrument_id=luid,
                    instrument_id_type="LusidInstrumentId",
                    quote_type="Price", field="mid"),
                effective_at=effective.isoformat()),
            metric_value=m.MetricValue(value=price, unit=currency))})


def value(portfolio, effective, metrics, currency, group_by=None):
    """Run the recipe over one portfolio and return the result as a DataFrame."""
    request = m.ValuationRequest(
        recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE),
        metrics=[m.AggregateSpec(key=k, op=op) for k, op in metrics],
        group_by=group_by or ["Instrument/default/Name"],
        report_currency=currency,
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=portfolio, portfolio_entity_type="SinglePortfolio")],
        valuation_schedule=m.ValuationSchedule(effective_at=effective.isoformat()))
    return pd.DataFrame(aggregation_api.get_valuation(valuation_request=request).data)


def transactions(portfolio, from_date, as_at):
    """The portfolio's own booked transactions over a date range, as a DataFrame."""
    txns = txn_portfolio_api.get_transactions(
        scope=SCOPE, code=portfolio,
        from_transaction_date=from_date.isoformat(),
        to_transaction_date=as_at.isoformat()).values
    if not txns:
        return pd.DataFrame(columns=["date", "type", "luid", "units", "consideration"])
    return pd.DataFrame([{
        "date": pd.Timestamp(t.transaction_date).strftime("%Y-%m-%d"),
        "type": t.type,
        "luid": t.instrument_uid,
        "units": t.units,
        "consideration": t.total_consideration.amount,
    } for t in txns]).sort_values(["date", "type"]).reset_index(drop=True)


SCOPE     = "MortgageOptionDemo"
RECIPE    = "mortgage-option-demo-recipe"
PORTFOLIO = "mortgage-option-demo-book"

TBA_ID    = "DEMO-FN55-TBA-01"
OPTION_ID = "DEMO-FN55-OPT-01"
DESC      = "Demo FNMA 5.50% TBA Call Option"
CURRENCY  = "USD"

AGENCY = "FNMA"
COUPON = 5.50
TENOR  = "30Y"

EXPIRY       = d(2025, 11, 21)
OPTION_START = EXPIRY - timedelta(days=72)      # option written ~72d ahead of TBA settlement
TBA_START    = OPTION_START
TBA_MATURITY = EXPIRY                            # TBA's own contract horizon is its settlement date
ASOF         = d(2025, 10, 15)                   # before EXPIRY -- never value an exercised option

OPTION_TYPE    = "Call"
STRIKE         = 100.00                          # points, par
DELIVERY_TYPE  = "Physical"
EXERCISE_TYPE  = "European"
PREMIUM_AMOUNT = 0.50                            # placeholder, see constraint above -- inert here

QUANTITY = 2_000_000.00
PRICE    = 0.75                                  # points, quoted mark
DENOM    = 100

print(f"{DESC}")
print(f"  underlying TBA {TBA_ID}: {AGENCY} {COUPON:.2f}% {TENOR}, settles {EXPIRY:%Y-%m-%d}")
print(f"  {EXERCISE_TYPE} {OPTION_TYPE} @ strike {STRIKE}, expiry {EXPIRY:%Y-%m-%d}")
print(f"  {QUANTITY:,.0f} units at {PRICE} points on {ASOF:%Y-%m-%d}")
print(f"  market value = {QUANTITY:,.0f} x {PRICE} / {DENOM} = {QUANTITY * PRICE / DENOM:,.2f} {CURRENCY}")

Demo FNMA 5.50% TBA Call Option
  underlying TBA DEMO-FN55-TBA-01: FNMA 5.50% 30Y, settles 2025-11-21
  European Call @ strike 100.0, expiry 2025-11-21
  2,000,000 units at 0.75 points on 2025-10-15
  market value = 2,000,000 x 0.75 / 100 = 15,000.00 USD


---
# 1. Instrument creation

We upsert the `ToBeAnnounced` underlying first, on its own. Then `mastered()` wraps its LUID so
it can go into the option's `underlying` field.

In [3]:
tba = m.ToBeAnnounced(
    instrument_type="ToBeAnnounced",
    start_date=TBA_START,
    maturity_date=TBA_MATURITY,
    dom_ccy=CURRENCY,
    agency=AGENCY,
    coupon=COUPON,
    tenor=TENOR)

TBA_LUID = upsert("tba", f"Underlying TBA {TBA_ID}", TBA_ID, tba)
print(f"Underlying TBA : {TBA_LUID}")

option = m.ToBeAnnouncedOption(
    instrument_type="ToBeAnnouncedOption",
    start_date=OPTION_START,
    expiry_date=EXPIRY,
    dom_ccy=CURRENCY,
    option_type=OPTION_TYPE,
    strike=STRIKE,
    delivery_type=DELIVERY_TYPE,
    underlying=mastered(TBA_LUID),
    exercise_type=EXERCISE_TYPE,
    premium=m.Premium(amount=PREMIUM_AMOUNT, currency=CURRENCY, date=OPTION_START))

OPTION_LUID = upsert("option", DESC, OPTION_ID, option)
print(f"Mortgage option : {OPTION_LUID}")

Underlying TBA : LUID_00003DG0


Mortgage option : LUID_00003DG1


---
# 2. Recipe

Under `SimpleStatic`, the position is priced off a quoted mark. The option's strike, expiry, and
underlying still describe what the instrument *is* -- under this model, they just don't feed into
the number.

In [4]:
recipes_api.upsert_configuration_recipe(
    upsert_recipe_request=m.UpsertRecipeRequest(
        configuration_recipe=m.ConfigurationRecipe(
            scope=SCOPE, code=RECIPE,
            description="Mortgage (TBA) option, marked",
            market=m.MarketContext(
                market_rules=[m.MarketDataKeyRule(
                    key="Quote.LusidInstrumentId.*", supplier="Lusid", data_scope=SCOPE,
                    quote_type="Price", field="mid", quote_interval="1Y")],
                options=m.MarketOptions(
                    default_supplier="Lusid",
                    default_instrument_code_type="LusidInstrumentId",
                    default_scope=SCOPE)),
            pricing=m.PricingContext(
                model_rules=[m.VendorModelRule(
                    supplier="Lusid", model_name="SimpleStatic",
                    instrument_type="ToBeAnnouncedOption")],
                options=m.PricingOptions(allow_partially_successful_evaluation=True)))))

print(f"Recipe: {SCOPE}/{RECIPE}")

Recipe: MortgageOptionDemo/mortgage-option-demo-recipe


---
# 3. Transaction types

Section 6 forecasts two events, `ExpiryEvent` and `OptionExercisePhysicalEvent` -- LUSID reads
`expiryDate` and `exerciseType` off the option and works out that these are the two ways the
position can end. Whether either one actually carries a *transaction* depends on a transaction
type of the right name being registered.

`Expiry` and `OptionExercisePhysical` are LUSID's own default transaction types, shipped for
exactly this purpose. We register them explicitly here rather than assume they're already
present, so the notebook still works if it runs against a domain where they've been removed.

In [5]:
txn_config_api = api(lusid.TransactionConfigurationApi)

TXN_TYPES = [
    ("Expiry",                 "Swaption/option expiry -- closes the expired position", -1),
    ("OptionExercisePhysical",  "Physically exercise an option",                         -1),
]

for txn_type, description, direction in TXN_TYPES:
    txn_config_api.set_transaction_type(
        source="default", type=txn_type, scope="default",
        transaction_type_request=m.TransactionTypeRequest(
            aliases=[m.TransactionTypeAlias(
                type=txn_type, description=description,
                transaction_class="Basic", transaction_roles="AllRoles", is_default=False)],
            movements=[m.TransactionTypeMovement(
                movement_types="StockMovement", side="Side1", direction=direction)]))
    print(f"{txn_type:<24} StockMovement Side1 {direction:+d}")

Expiry                   StockMovement Side1 -1


OptionExercisePhysical   StockMovement Side1 -1


---
# 4. Portfolio and transactions

One `Buy` transaction books the option position -- no principal changes hands beyond its own
traded price.

`instrumentEventConfiguration`, set below via `recipe=RECIPE`, tells LUSID which recipe to use
when forecasting this book's events. It can only be set at portfolio creation, which is why
`recreate_portfolio()` deletes and recreates the portfolio rather than just updating it. Skip this
setting and section 6 will quietly come back with zero events -- no error, just nothing there.

In [6]:
recreate_portfolio(PORTFOLIO, "Mortgage Option Demo Book", CURRENCY, d(2025, 1, 1), recipe=RECIPE)

txn_portfolio_api.upsert_transactions(
    scope=SCOPE, code=PORTFOLIO,
    transaction_request=[m.TransactionRequest(
        transaction_id="BUY-MTGOPT",
        type="Buy",
        instrument_identifiers={"Instrument/default/LusidInstrumentId": OPTION_LUID},
        transaction_date=OPTION_START.isoformat(),
        settlement_date=OPTION_START.isoformat(),
        units=QUANTITY,
        transaction_price=m.TransactionPrice(price=0.0, type="Price"),
        total_consideration=m.CurrencyAndAmount(amount=0.0, currency=CURRENCY),
        source="default")])

display(transactions(PORTFOLIO, OPTION_START, OPTION_START))

Recreated MortgageOptionDemo/mortgage-option-demo-book


,date,type,luid,units,consideration
0,2025-09-10,Buy,LUID_00003DG1,"2,000,000.00",0.00


---
# 5. Valuation

One quote, at the option's own price and expressed per unit, valued before expiry.

In [7]:
upsert_price(OPTION_LUID, PRICE / DENOM, ASOF, CURRENCY)

METRICS = [("Instrument/default/Name", "Value"),
           ("Holding/default/Units",   "Sum"),
           ("Valuation/CleanPV",       "Sum")]

result = value(PORTFOLIO, ASOF, METRICS, CURRENCY)
display(result)

pv = result.loc[result["Instrument/default/Name"] == DESC, "Sum(Valuation/CleanPV)"].iloc[0]
print(f"LUSID CleanPV {pv:,.2f}  vs  quoted mark {QUANTITY * PRICE / DENOM:,.2f}")

,Instrument/default/Name,Sum(Holding/default/Units),Sum(Valuation/CleanPV)
0,Demo FNMA 5.50% TBA Call Option,"2,000,000.00","15,000.00"


LUSID CleanPV 15,000.00  vs  quoted mark 15,000.00


---
# 6. Instrument events

`expiryDate` and `exerciseType` on the option are enough for LUSID to work out both ways this
position can end. Nothing below actually posts an event -- `query_applicable_instrument_events`
only forecasts what could happen.

## 6a. Applicable events

A window spanning `EXPIRY` should surface both of the option's applicable event types.

In [8]:
events_api      = api(lusid.InstrumentEventsApi)
event_types_api = api(lusid.InstrumentEventTypesApi)

WINDOW_START = OPTION_START
WINDOW_END   = EXPIRY + timedelta(days=10)

applicable = events_api.query_applicable_instrument_events(
    query_applicable_instrument_events_request=m.QueryApplicableInstrumentEventsRequest(
        window_start=WINDOW_START.isoformat(),
        window_end=WINDOW_END.isoformat(),
        effective_at=WINDOW_END.isoformat(),
        portfolio_entity_ids=[m.PortfolioEntityId(
            scope=SCOPE, code=PORTFOLIO, portfolio_entity_type="SinglePortfolio")],
        forecasting_recipe_id=m.ResourceId(scope=SCOPE, code=RECIPE))).values

print(f"{len(applicable)} applicable event(s):\n")
display(pd.DataFrame([{
    "event type": ev.instrument_event_type,
    "instrument": DESC if ev.lusid_instrument_id == OPTION_LUID else ev.lusid_instrument_id,
    "eligible balance": ev.eligible_balance,
    "status": ev.instrument_event_status,
} for ev in applicable]))

2 applicable event(s):



,event type,instrument,eligible balance,status
0,OptionExercisePhysicalEvent,Demo FNMA 5.50% TBA Call Option,"2,000,000.00",Active
1,ExpiryEvent,Demo FNMA 5.50% TBA Call Option,"2,000,000.00",Active


## 6b. The transactions each event carries

Every `ApplicableInstrumentEvent` carries whatever transactions its template produced, drawing on
the transaction types registered in section 3.

In [9]:
rows = []
for ev in applicable:
    for txn in (ev.transactions or []):
        rows.append({
            "event": ev.instrument_event_type,
            "txn type": getattr(txn, "type", None),
            "units": getattr(txn, "units", None),
            "price": getattr(getattr(txn, "transaction_price", None), "price", None),
            "consideration": getattr(getattr(txn, "total_consideration", None), "amount", None),
        })

if rows:
    display(pd.DataFrame(rows))

covered = {r["event"] for r in rows}
for ev in applicable:
    if ev.instrument_event_type not in covered:
        print(f"{ev.instrument_event_type}: 0 transactions.")

,event,txn type,units,price,consideration
0,ExpiryEvent,Expiry,"2,000,000.00",0.00,0.00


OptionExercisePhysicalEvent: 0 transactions.


## 6c. Voluntary events and the forecasting endpoint

`ExpiryEvent` and `OptionExercisePhysicalEvent` differ in `supportedParticipationTypes`, the
field that decides whether either event can produce a transaction here, regardless of which
transaction types happen to be registered.

In [10]:
for event_type in ("ExpiryEvent", "OptionExercisePhysicalEvent"):
    sd = event_types_api.get_transaction_template_specification(instrument_event_type=event_type)
    print(f"{event_type:<28} participation={sd.supported_participation_types}  "
          f"elections={[e.election_type for e in (sd.supported_election_types or [])]}")

ExpiryEvent                  participation=['Mandatory']  elections=[]


OptionExercisePhysicalEvent  participation=['Voluntary']  elections=['OptionExerciseElection']


`ExpiryEvent` is `Mandatory`: there's only one possible outcome, so the registered `Expiry`
transaction type is all the template needs to produce a transaction.

`OptionExercisePhysicalEvent` is `Voluntary` and carries a required `OptionExerciseElection`. Its
outcome depends on an election, and `query_applicable_instrument_events` has no way to supply one
-- that endpoint only forecasts. So this event type won't produce a transaction here no matter
which transaction types are registered; that comes down to `Voluntary` participation, not
anything missing from this notebook's setup. `OptionExercisePhysical` still gets registered in
section 3 for the same reason as `Expiry`: it's LUSID's own default transaction type for this
event, and it's what a real election would use once one comes through a workflow that actually
carries elections.

---
# Summary

1. A mortgage option is a `ToBeAnnouncedOption` whose `underlying` points at a `ToBeAnnounced` --
   upsert that first, on its own, then wrap it with `mastered()`.
2. `premium` takes a placeholder value; under `SimpleStatic`, used here, it plays no part in the
   position's price -- that comes from its own quoted mark instead.
3. It's reported at that quoted mark under `SimpleStatic`.
4. `ExpiryEvent` and `OptionExercisePhysicalEvent` are both forecastable once the portfolio's
   `instrumentEventConfiguration` names a recipe. Getting an actual *transaction* out of either
   one also needs a transaction type of the right name registered -- `Expiry` and
   `OptionExercisePhysical` are LUSID's own default transaction types for this.
5. `ExpiryEvent` is `Mandatory`, so registering `Expiry` is enough to see a transaction show up.
   `OptionExercisePhysicalEvent` is `Voluntary` and needs an `OptionExerciseElection` that the
   forecasting endpoint in section 6 has no way to supply, so no amount of transaction type
   registration will make it show a transaction here. That's a limit of forecasting voluntary
   events, not a missing transaction type.

In [11]:
print(f"Scope      : {SCOPE}")
print(f"Portfolio  : {SCOPE}/{PORTFOLIO}")
print(f"Recipe     : {SCOPE}/{RECIPE}")
print(f"Underlying : {TBA_LUID}")
print(f"Instrument : {OPTION_LUID}")

Scope      : MortgageOptionDemo
Portfolio  : MortgageOptionDemo/mortgage-option-demo-book
Recipe     : MortgageOptionDemo/mortgage-option-demo-recipe
Underlying : LUID_00003DG0
Instrument : LUID_00003DG1
